In [1]:
import pandas as pd
from micom.workflows import build
from micom import load_pickle
from micom.qiime_formats import load_qiime_model_db, load_qiime_medium
from micom.workflows import grow
import pickle
from micom.viz import *
from micom.workflows import tradeoff

In [2]:
taxonomy_names = {'B_caccae': 'Bacteroides caccae',
                'B_cellulosilyticus_WH2': 'Bacteroides cellulosilyticus',
                'B_ovatus': 'Bacteroides ovatus',
                'B_thetaiotaomicron': 'Bacteroides thetaiotaomicron',
                'B_uniformis': 'Bacteroides uniformis',
                'B_vulgatus': 'Phocaeicola vulgatus',
                'C_aerofaciens': 'Collinsella aerofaciens',
                'C_scindens': 'Clostridium scindens',
                'C_spiroforme': 'Clostridium spiroforme',
                'D_longicatena': 'Dorea longicatena',
                'P_distasonis': 'Parabacteroides distasonis',
                'R_obeum': 'Blautia obeum',
                }

model_ids = {'B_caccae': 'Bacteroides_caccae_ATCC_43185.xml',
                'B_cellulosilyticus_WH2': 'Bacteroides_cellulosilyticus_WH2.xml',
                'B_ovatus': 'Bacteroides_ovatus_ATCC_8483.xml',
                'B_thetaiotaomicron': 'Bacteroides_thetaiotaomicron_VPI_5482.xml',
                'B_uniformis': 'Bacteroides_uniformis_ATCC_8492.xml',
                'B_vulgatus': 'Bacteroides_vulgatus_ATCC_8482.xml',
                'C_aerofaciens': 'Collinsella_aerofaciens_ATCC_25986.xml',
                'C_scindens': 'Clostridium_scindens_ATCC_35704.xml',
                'C_spiroforme': 'Clostridium_spiroforme_DSM_1552.xml',
                # 'D_longicatena': 'Dorea_longicatena',
                'P_distasonis': 'Parabacteroides_distasonis_ATCC_8503.xml',
                'R_obeum': 'Blautia_obeum_ATCC_29174.xml'}

In [3]:
lf0_reads = pd.read_table('../../ecological-modeling/mcnulty-datasets/LF0/reads.tsv', index_col=0, sep='\t')
lf0_reads = lf0_reads.loc[:,['-12' in col for col in lf0_reads.columns]].copy()
lf0_reads.reset_index(inplace=True)
lf0_reads = lf0_reads.melt(id_vars=['name'], var_name='sample', value_name='reads')
lf0_reads['sample'] = lf0_reads['sample'].str.replace('-12', '')
lf0_reads['species'] = lf0_reads['name'].map(taxonomy_names)
lf0_reads.columns = ['id', 'sample_id', 'abundance', 'species']
lf0_reads['genus'] = lf0_reads['species'].str.split(' ').str[0]
lf0_reads = lf0_reads[lf0_reads['id'] != 'D_longicatena']
lf0_reads['file'] = 'models/Bacteria/' + lf0_reads['id'].map(model_ids)
lf0_reads['sample_id'] = 'mouse_' + lf0_reads['sample_id']
lf0_reads['diet'] = 'LF/HPP'
lf0_reads.to_csv('lf0_data.tsv', sep='\t', index=False)
lf0_reads

,id,sample_id,abundance,species,genus,file,diet
0,B_caccae,mouse_01,9829.0,Bacteroides caccae,Bacteroides,models/Bacteria/Bacteroides_caccae_ATCC_43185.xml,LF/HPP
1,B_cellulosilyticus_WH2,mouse_01,44957.8,Bacteroides cellulosilyticus,Bacteroides,models/Bacteria/Bacteroides_cellulosilyticus_W...,LF/HPP
2,B_ovatus,mouse_01,14105.3,Bacteroides ovatus,Bacteroides,models/Bacteria/Bacteroides_ovatus_ATCC_8483.xml,LF/HPP
3,B_thetaiotaomicron,mouse_01,9462.1,Bacteroides thetaiotaomicron,Bacteroides,models/Bacteria/Bacteroides_thetaiotaomicron_V...,LF/HPP
4,B_uniformis,mouse_01,3552.0,Bacteroides uniformis,Bacteroides,models/Bacteria/Bacteroides_uniformis_ATCC_849...,LF/HPP
...,...,...,...,...,...,...,...
78,C_aerofaciens,mouse_07,699.0,Collinsella aerofaciens,Collinsella,models/Bacteria/Collinsella_aerofaciens_ATCC_2...,LF/HPP
79,C_scindens,mouse_07,1469.8,Clostridium scindens,Clostridium,models/Bacteria/Clostridium_scindens_ATCC_3570...,LF/HPP
80,C_spiroforme,mouse_07,19.5,Clostridium spiroforme,Clostridium,models/Bacteria/Clostridium_spiroforme_DSM_155...,LF/HPP
82,P_distasonis,mouse_07,2156.1,Parabacteroides distasonis,Parabacteroides,models/Bacteria/Parabacteroides_distasonis_ATC...,LF/HPP


In [4]:
manifest_lf0 = build(
    lf0_reads,  
    model_db=None,
    out_folder='models/Community/lf0', 
    cutoff=0.000001, 
    threads=4
    )
manifest_lf0